# Notebook 07d — Calibration Health Flag for SCTS-v2

**Purpose:** Augment SCTS-v2 trust scores with a per-(model, predicted_class) **calibration health flag** that warns users when SCTS may be unreliable due to underlying calibration failures.

**Motivation:** SCTS-v2 inherits calibration quality. When the underlying calibrator fails on rare classes (e.g., NSL R2L: 90.6% of R2L samples have calibrated p_hat(R2L) = 0), SCTS produces high values for confidently-wrong predictions (e.g., NSL R2L: SCTS=77.2, accuracy=6.7%). A user looking at a 77/100 score has no way to know it's degenerate. The health flag makes this visible at decision time.

## Three signals

Per (dataset, model, predicted_class):

| Signal | Green | Amber | Red |
|---|---|---|---|
| **Threshold usable** | threshold ∈ (0.05, 0.95) | threshold ∈ [0.95, 0.999) or [0.001, 0.05] | threshold ≥ 0.999 or < 0.001 |
| **Cliff fraction** | < 5% | 5–20% | ≥ 20% |
| **Calibration support** | n_calib ≥ 100 | n_calib ∈ [30, 100) | n_calib < 30 (fallback) |

**Overall flag = worst (most-red) of the three signals.**

## What each signal catches

- **Signal 1 (threshold usable):** Catches both degenerate-too-loose (NSL pattern: threshold = 1.0) and degenerate-too-strict (CIC tree pattern: threshold ≈ 0.005). In either degeneracy, c₃ fails to discriminate.
- **Signal 2 (cliff fraction):** Catches the underlying calibration failure that produced the threshold problem. Fraction of calibration samples with score = `1 - p_hat(y_true) ≥ 0.95` (calibrator assigns near-zero probability to true class).
- **Signal 3 (calibration support):** Catches insufficient calibration data for stable per-class quantile estimation. Already used by Mondrian fallback rule.

## Outputs

- `results/tables/scts_v2_calib_health.csv` (90 rows: 18 ds-models × 5 predicted_classes)
- `results/tables/scts_v2_canonical_with_health.csv` (the original 18,000 SCTS rows augmented with `calib_health` per sample, joined by predicted_class)
- `results/tables/scts_v2_health_summary.json` (counts of green/amber/red per dataset, summary stats)

**Time estimate:** ~3 minutes total.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

print(f'✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from datetime import datetime

DATASETS = ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
ARCHITECTURES = ['rf', 'xgb', 'dnn']
VARIANTS = ['5class_cw', '5class_smote']
MODELS_PER_DATASET = [f'{a}_{v}' for v in VARIANTS for a in ARCHITECTURES]
CLASS_NAMES_5 = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']

# Thresholds for the three signals
# Signal 1: threshold value range
T_GREEN_LO, T_GREEN_HI = 0.05, 0.95   # usable range
T_RED_HI = 0.999                       # degenerate-loose
T_RED_LO = 0.001                       # degenerate-strict

# Signal 2: cliff fraction (fraction of calib samples with score >= 0.95)
CLIFF_GREEN_HI = 0.05  # < 5% is green
CLIFF_RED_LO = 0.20    # >= 20% is red
CLIFF_THRESH = 0.95    # score threshold defining a 'cliff' sample

# Signal 3: calibration support
N_GREEN_LO = 100
N_RED_HI = 30  # < 30 = red (matches Mondrian fallback rule)

print(f'Loaded constants. {len(DATASETS)} datasets x {len(MODELS_PER_DATASET)} models = {len(DATASETS) * len(MODELS_PER_DATASET)} model-dataset cells')
print(f'5 predicted classes per model => {len(DATASETS) * len(MODELS_PER_DATASET) * 5} (ds, model, pred_class) combinations expected')

Loaded constants. 3 datasets x 6 models = 18 model-dataset cells
5 predicted classes per model => 90 (ds, model, pred_class) combinations expected


## 2. Load existing SCTS metadata (signals 1 and 3 are already computed in 07c)

In [3]:
with open(Path(REPO) / 'results' / 'tables' / 'scts_v2_summary.json') as f:
    scts_summary = json.load(f)

conformal_thresholds = scts_summary['conformal_thresholds']
print(f'Loaded conformal thresholds for {len(conformal_thresholds)} (dataset, model) cells')
print(f'Sample entry (nsl_kdd_v2/rf_5class_cw):')
print(json.dumps(conformal_thresholds.get('nsl_kdd_v2/rf_5class_cw', {}), indent=2))

Loaded conformal thresholds for 18 (dataset, model) cells
Sample entry (nsl_kdd_v2/rf_5class_cw):
{
  "marginal_threshold_alpha_0.05": 1.0,
  "mondrian_thresholds_alpha_0.05": {
    "0": 1.0,
    "1": 0.48000115624718354,
    "2": 1.0,
    "3": 0.49354748735004383,
    "4": 1.0
  },
  "fallback_classes": [
    4
  ],
  "n_calib_per_predicted_class": {
    "0": 13334,
    "1": 6327,
    "2": 1771,
    "3": 108,
    "4": 4
  },
  "empirical_coverage_on_canonical_mondrian": 0.975,
  "n_conformal_calibration_samples": 21544
}


## 3. Compute Signal 2 (cliff fractions) per (dataset, model, predicted_class)

In [ ]:
cliff_data = {}  # (ds, model, pred_class) -> fraction of calib samples with score >= 0.95

for ds in DATASETS:
    print(f'\n=== {ds} ===')
    canonical_eval_idx = np.load(Path(REPO) / 'shap_values' / ds / 'canonical_eval_idx.npy')
    y_test_full = np.load(f'{REPO}/data/processed/{ds}/y_test_5class.npy')
    n_test = len(y_test_full)
    mask_calib = np.ones(n_test, dtype=bool)
    mask_calib[canonical_eval_idx] = False
    y_calib = y_test_full[mask_calib]

    for model_name in MODELS_PER_DATASET:
        prob_path = Path(REPO) / 'calibrators' / ds / f'{model_name}_test_proba_calibrated.npy'
        calibrated_proba_full = np.load(prob_path)
        calibrated_proba_calib = calibrated_proba_full[mask_calib]

        # Per-sample nonconformity score using TRUE class
        scores = 1.0 - calibrated_proba_calib[np.arange(len(y_calib)), y_calib]
        # And per-sample predicted class
        y_pred_calib = calibrated_proba_calib.argmax(axis=1)

        # Cliff fraction per PREDICTED class
        for pred_cls in range(5):
            mask_pred = y_pred_calib == pred_cls
            n_in_pred = int(mask_pred.sum())
            if n_in_pred == 0:
                cliff_frac = float('nan')
            else:
                cliff_frac = float((scores[mask_pred] >= CLIFF_THRESH).mean())
            cliff_data[(ds, model_name, pred_cls)] = {
                'cliff_fraction': cliff_frac,
                'n_pred_class_in_calib': n_in_pred,
            }

        # One-line summary per model
        cliffs = [cliff_data[(ds, model_name, c)]['cliff_fraction'] for c in range(5)]
        cliff_str = ' '.join(f'{c:.2f}' if not np.isnan(c) else 'NaN' for c in cliffs)
        print(f'  {model_name:<22} cliff_fracs=[{cliff_str}]')

print(f'\n✓ Computed cliff fractions for {len(cliff_data)} (ds, model, pred_class) combinations')


=== nsl_kdd_v2 ===
  rf_5class_cw           cliff_fracs=[0.28 0.03 0.12 0.00 0.50]


## 4. Compute the three signals and overall flag per (ds, model, pred_class)

In [ ]:
def flag_threshold(thresh):
    """Signal 1: threshold usable range."""
    if thresh >= T_RED_HI or thresh < T_RED_LO:
        return 'red'
    if thresh >= T_GREEN_HI or thresh <= T_GREEN_LO:
        return 'amber'
    return 'green'

def flag_cliff(cliff_frac):
    """Signal 2: cliff fraction."""
    if np.isnan(cliff_frac):
        # No samples predicted this class on calib set — unable to verify
        return 'red'
    if cliff_frac >= CLIFF_RED_LO:
        return 'red'
    if cliff_frac >= CLIFF_GREEN_HI:
        return 'amber'
    return 'green'

def flag_support(n_calib):
    """Signal 3: calibration support."""
    if n_calib < N_RED_HI:
        return 'red'
    if n_calib < N_GREEN_LO:
        return 'amber'
    return 'green'

def combine_flags(*flags):
    """Worst (most-red) of the provided flags."""
    if 'red' in flags:
        return 'red'
    if 'amber' in flags:
        return 'amber'
    return 'green'

# Build the health table
health_records = []

for ds in DATASETS:
    for model_name in MODELS_PER_DATASET:
        key = f'{ds}/{model_name}'
        meta = conformal_thresholds[key]

        mondrian_thresh = meta['mondrian_thresholds_alpha_0.05']  # {class_idx_str: threshold}
        fallback_classes = set(meta['fallback_classes'])
        n_per_class = meta['n_calib_per_predicted_class']  # {class_idx_str: n}

        for pred_cls in range(5):
            cls_str = str(pred_cls)
            thresh = mondrian_thresh[cls_str]
            n_calib = n_per_class[cls_str]
            cliff_info = cliff_data[(ds, model_name, pred_cls)]
            cliff_frac = cliff_info['cliff_fraction']

            sig_thresh = flag_threshold(thresh)
            sig_cliff = flag_cliff(cliff_frac)
            sig_support = flag_support(n_calib)
            overall = combine_flags(sig_thresh, sig_cliff, sig_support)

            health_records.append({
                'dataset': ds,
                'model': model_name,
                'predicted_class_idx': pred_cls,
                'predicted_class': CLASS_NAMES_5[pred_cls],
                'mondrian_threshold': thresh,
                'is_fallback': pred_cls in fallback_classes,
                'n_calib': n_calib,
                'cliff_fraction': cliff_frac,
                'signal_threshold': sig_thresh,
                'signal_cliff': sig_cliff,
                'signal_support': sig_support,
                'calib_health': overall,
            })

df_health = pd.DataFrame(health_records)
print(f'✓ Built health table: {len(df_health)} rows (expected {len(DATASETS) * len(MODELS_PER_DATASET) * 5})')
print(f'\nFlag distribution:')
print(df_health['calib_health'].value_counts())
print(f'\nFlag distribution per dataset:')
print(df_health.groupby(['dataset', 'calib_health']).size().unstack(fill_value=0))

## 5. Inspect the health table

In [ ]:
print('=' * 90)
print('CALIBRATION HEALTH TABLE — per (dataset, model, predicted_class)')
print('=' * 90)
for ds in DATASETS:
    print(f'\n--- {ds} ---')
    sub = df_health[df_health['dataset'] == ds]
    for model_name in MODELS_PER_DATASET:
        sub_m = sub[sub['model'] == model_name]
        flag_str = ' '.join(
            f'{row["predicted_class"][:3]}={row["calib_health"][0].upper()}'
            for _, row in sub_m.iterrows()
        )
        print(f'  {model_name:<22} {flag_str}')

# Save
out_path = Path(REPO) / 'results' / 'tables' / 'scts_v2_calib_health.csv'
df_health.to_csv(out_path, index=False)
print(f'\n✓ Saved: {out_path.name} ({out_path.stat().st_size} bytes)')

## 6. Augment per-sample SCTS CSV with calib_health

In [ ]:
df_scts = pd.read_csv(Path(REPO) / 'results' / 'tables' / 'scts_v2_canonical.csv')
print(f'Loaded {len(df_scts)} per-sample SCTS rows')

# Join: each sample's predicted class -> flag from health table
flag_lookup = df_health.set_index(['dataset', 'model', 'predicted_class_idx'])['calib_health'].to_dict()
df_scts['calib_health'] = df_scts.apply(
    lambda row: flag_lookup.get((row['dataset'], row['model'], row['pred_class']), 'unknown'),
    axis=1,
)

# Sanity check
n_unknown = (df_scts['calib_health'] == 'unknown').sum()
print(f'Unknown flags (should be 0): {n_unknown}')
print(f'\nFlag distribution across 18,000 per-sample rows:')
print(df_scts['calib_health'].value_counts())

# Save augmented CSV
out_path = Path(REPO) / 'results' / 'tables' / 'scts_v2_canonical_with_health.csv'
df_scts.to_csv(out_path, index=False)
print(f'\n✓ Saved: {out_path.name} ({out_path.stat().st_size} bytes)')

## 7. The key validation: does the flag actually identify the broken cases?

In [ ]:
print('=' * 90)
print('VALIDATION: SCTS-vs-accuracy relationship, broken out by health flag')
print('=' * 90)
print('\nIf the flag works:')
print('  - Green-flagged samples: SCTS should correlate strongly with correctness')
print('  - Red-flagged samples: SCTS may NOT correlate with correctness (the broken regime)')
print('\n' + '-' * 70)

for flag in ['green', 'amber', 'red']:
    sub = df_scts[df_scts['calib_health'] == flag]
    if len(sub) == 0:
        print(f'\n{flag.upper()}: 0 samples')
        continue

    # Pearson per model in this flag group
    per_model = []
    for (ds, m), g in sub.groupby(['dataset', 'model']):
        if g['scts'].std() > 1e-9 and len(g) > 5:
            p = np.corrcoef(g['scts'], g['correct'])[0, 1]
            per_model.append({'ds': ds, 'model': m, 'n': len(g), 'pearson': p})

    print(f'\n{flag.upper()} ({len(sub):>5} samples):')
    print(f'  Mean SCTS: {sub["scts"].mean():.1f}')
    print(f'  Mean accuracy: {sub["correct"].mean():.3f}')
    if per_model:
        per_model_df = pd.DataFrame(per_model)
        print(f'  Per-model Pearson (n_models={len(per_model_df)}):')
        print(f'    Mean: {per_model_df["pearson"].mean():+.3f}')
        print(f'    Median: {per_model_df["pearson"].median():+.3f}')
        print(f'    Min: {per_model_df["pearson"].min():+.3f}, Max: {per_model_df["pearson"].max():+.3f}')
        print(f'    Models with positive Pearson: {(per_model_df["pearson"] > 0).sum()}/{len(per_model_df)}')

    # Per-class breakdown
    pc = sub.groupby('true_class').agg({
        'scts': 'mean',
        'correct': 'mean',
        'sample_position': 'count',
    }).round(3).rename(columns={'sample_position': 'n'})
    pc.index = [CLASS_NAMES_5[i] for i in pc.index]
    print(f'  Per-class:')
    print(pc.to_string().replace('\n', '\n    '))

# The CRITICAL test: NSL R2L samples
print('\n' + '=' * 90)
print('CRITICAL: NSL R2L samples — should be RED-flagged')
print('=' * 90)
nsl_r2l = df_scts[(df_scts['dataset'] == 'nsl_kdd_v2') & (df_scts['true_class'] == 3)]
print(f'\nNSL R2L (true class) samples: n={len(nsl_r2l)}')
print(f'  Mean SCTS: {nsl_r2l["scts"].mean():.1f}')
print(f'  Mean accuracy: {nsl_r2l["correct"].mean():.3f}')
print(f'  Flag distribution:')
print(nsl_r2l['calib_health'].value_counts())
print(f'\n  Per (model, pred_class) for these R2L true-class samples:')
for (m, pc, ch), g in nsl_r2l.groupby(['model', 'pred_class', 'calib_health']):
    print(f'    {m:<22} pred={CLASS_NAMES_5[pc]:<7} flag={ch:<5} n={len(g)} mean_scts={g["scts"].mean():.1f} acc={g["correct"].mean():.3f}')

## 8. Save summary JSON

In [ ]:
def to_json_safe(obj):
    if isinstance(obj, dict):
        return {('|'.join(str(x) for x in k) if isinstance(k, tuple) else (k if isinstance(k, (str, int, float, bool)) or k is None else str(k))): to_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_json_safe(x) for x in obj]
    if isinstance(obj, (np.bool_, np.generic)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

# Aggregate per-flag SCTS and Pearson
summary_by_flag = {}
for flag in ['green', 'amber', 'red']:
    sub = df_scts[df_scts['calib_health'] == flag]
    if len(sub) == 0:
        summary_by_flag[flag] = {'n_samples': 0}
        continue
    per_model_pearson = []
    for (ds, m), g in sub.groupby(['dataset', 'model']):
        if g['scts'].std() > 1e-9 and len(g) > 5:
            per_model_pearson.append(float(np.corrcoef(g['scts'], g['correct'])[0, 1]))
    summary_by_flag[flag] = {
        'n_samples': int(len(sub)),
        'mean_scts': float(sub['scts'].mean()),
        'mean_accuracy': float(sub['correct'].mean()),
        'n_models_with_data': len(per_model_pearson),
        'mean_pearson_per_model': float(np.mean(per_model_pearson)) if per_model_pearson else None,
        'median_pearson_per_model': float(np.median(per_model_pearson)) if per_model_pearson else None,
    }

summary = {
    'timestamp': datetime.now().isoformat(),
    'method': 'three_signal_health_flag_per_predicted_class',
    'signal_thresholds': {
        'threshold_green_range': [T_GREEN_LO, T_GREEN_HI],
        'threshold_red_high': T_RED_HI,
        'threshold_red_low': T_RED_LO,
        'cliff_green_max': CLIFF_GREEN_HI,
        'cliff_red_min': CLIFF_RED_LO,
        'cliff_score_threshold': CLIFF_THRESH,
        'support_green_min': N_GREEN_LO,
        'support_red_max': N_RED_HI,
    },
    'overall_flag_counts': {
        'class_level': df_health['calib_health'].value_counts().to_dict(),
        'sample_level': df_scts['calib_health'].value_counts().to_dict(),
    },
    'per_dataset_class_level_counts': df_health.groupby(['dataset', 'calib_health']).size().unstack(fill_value=0).to_dict(),
    'summary_by_flag': summary_by_flag,
}

out_path = Path(REPO) / 'results' / 'tables' / 'scts_v2_health_summary.json'
with open(out_path, 'w') as f:
    json.dump(to_json_safe(summary), f, indent=2, default=str)
print(f'✓ Saved: {out_path.name}')
print(f'\nSummary contents:')
print(json.dumps(to_json_safe(summary), indent=2, default=str)[:3000])
print('... (truncated)')

## 9. Commit

In [ ]:
os.chdir(REPO)
!git add notebooks/07d_scts_calib_health.ipynb
!git add results/tables/scts_v2_calib_health.csv
!git add results/tables/scts_v2_canonical_with_health.csv
!git add results/tables/scts_v2_health_summary.json
!git status --short
!git commit -m 'Notebook 07d: SCTS-v2 calibration health flag (3-signal per (model, predicted_class), green/amber/red, flags NSL R2L predictions as red, validates Mondrian numbers + augments per-sample CSV with health column)'
!git push origin main